# Statistical Analysis: Multimodal Image Recognition Evaluation

**Research context:** Master's thesis comparing 8 multimodal vision model approaches for Norwegian cultural heritage/tourism applications.

**Dataset:** 150 fully evaluated tourist/travel photographs, 13 approach evaluations per image (1,950 total evaluations), 7,291 location claims, 10,796 entity claims.

**Approaches:**
- GPT Vision Direct (strong/fast) ± GPS
- GPT Vision + Search (strong/fast) ± GPS
- Gemini Vision (strong/fast) ± GPS
- Google Cloud Vision (no GPS, special case)

---

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations

from scipy import stats
from scipy.stats import (
    friedmanchisquare, wilcoxon, mannwhitneyu, 
    chi2_contingency, spearmanr, kruskal
)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Visualization defaults
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
sns.set_style('whitegrid')

# ── Consistent color scheme ──
APPROACH_COLORS = {
    'GPT Vision Direct (strong)':    '#1f77b4',
    'GPT Vision Direct (fast)':      '#aec7e8',
    'GPT Vision + Search (strong)':  '#ff7f0e',
    'GPT Vision + Search (fast)':    '#ffbb78',
    'Gemini Vision (strong)':        '#2ca02c',
    'Gemini Vision (fast)':          '#98df8a',
    'Google Cloud Vision':           '#d62728',
}

# Short labels for plots
APPROACH_SHORT = {
    'GPT Vision Direct (strong)':    'GPT-D (S)',
    'GPT Vision Direct (fast)':      'GPT-D (F)',
    'GPT Vision + Search (strong)':  'GPT+S (S)',
    'GPT Vision + Search (fast)':    'GPT+S (F)',
    'Gemini Vision (strong)':        'Gem (S)',
    'Gemini Vision (fast)':          'Gem (F)',
    'Google Cloud Vision':           'GCV',
}

APPROACH_ORDER = [
    'GPT Vision Direct (strong)', 'GPT Vision Direct (fast)',
    'GPT Vision + Search (strong)', 'GPT Vision + Search (fast)',
    'Gemini Vision (strong)', 'Gemini Vision (fast)',
    'Google Cloud Vision',
]

# GPS line styles
GPS_STYLE = {True: '-', False: '--'}
GPS_LABEL = {True: 'GPS', False: 'No GPS'}

# Correctness color map
CORR_CMAP = {1.0: '#2ca02c', 0.5: '#bcbd22', 0: '#999999', -0.5: '#ff7f0e', -1.0: '#d62728'}

ALPHA = 0.05  # significance level

print("Setup complete.")


## Section 0: Data Loading

In [ ]:
# ── Paths ──
BASE = Path('/Users/andreas/Documents/Dokumenter – MacBook Air/UIO/master/Thesis/Project/testing/Image_search')
EVAL_PATH = BASE / 'Evaluation' / 'human_evaluation.json'
TIMING_PATH = BASE / 'results' / 'v1 batch' / 'batch_test_results_2026-01-26.json'

# ── Load evaluation data ──
with open(EVAL_PATH) as f:
    raw = json.load(f)

print(f"Total images in file: {len(raw['images'])}")
print(f"Metadata: {raw['metadata']}")

# Filter to complete images only
images = {k: v for k, v in raw['images'].items() if v['status'] == 'complete'}
print(f"Complete images: {len(images)}")

# ── Load timing data ──
with open(TIMING_PATH) as f:
    timing_raw = json.load(f)

print(f"Timing records: {len(timing_raw['results'])}")


## Scoring Methodology

### Entity Score
Combines correctness, specificity achievement ratio, and importance weighting.

### Location Score  
`correctness × (specificity / 5.0)` — rewarding both accuracy and geographic precision.

### Overall Score
Weighted: 60% entity + 30% location + 10% description. GCV uses 67% entity + 33% location (no description).

In [ ]:
# ── Scoring functions ──

def entity_score(correctness, specificity, max_specificity, importance):
    """Score a single entity claim."""
    if correctness is None or correctness == 0:
        return 0.0
    
    # Specificity achievement ratio (allow slight bonus for exceeding max)
    if max_specificity and max_specificity > 0:
        spec_ratio = min(1.2, specificity / max_specificity)
    else:
        spec_ratio = 1.0 if specificity is not None else 0.5
    
    # Importance weights
    imp_weights = {'high': 1.5, 'medium': 1.0, 'low': 0.5}
    imp_w = imp_weights.get(importance, 1.0)
    
    score = correctness * spec_ratio * imp_w
    return max(-1.5, min(1.5, score))


def entity_score_unlinked(correctness, specificity):
    """Score for entity claims not linked to GT (scene/bonus/incorrect)."""
    if correctness is None or correctness == 0:
        return 0.0
    if specificity is None:
        specificity = 1
    # No importance weighting, use raw specificity / 5
    return correctness * (specificity / 5.0)


def location_score(correctness, specificity):
    """Score a single location claim."""
    if correctness is None or correctness == 0:
        return 0.0
    return correctness * (specificity / 5.0)


def overall_score(entity_mean, location_mean, desc_score, is_gcv=False):
    """Weighted overall score per approach × image."""
    if is_gcv:
        # No description score for GCV
        if entity_mean is not None and location_mean is not None:
            return 0.67 * entity_mean + 0.33 * location_mean
        elif entity_mean is not None:
            return entity_mean
        elif location_mean is not None:
            return location_mean
        return 0.0
    
    # Normalize desc_score from 1-5 to 0-1
    desc_norm = ((desc_score - 1) / 4.0) if desc_score is not None else 0.5
    
    e = entity_mean if entity_mean is not None else 0.0
    l = location_mean if location_mean is not None else 0.0
    
    return 0.60 * e + 0.30 * l + 0.10 * desc_norm


print("Scoring functions defined.")


## Data Processing Pipeline
Flatten the nested JSON into analysis-ready DataFrames.

In [ ]:
# ── Build entity claims DataFrame ──
entity_rows = []
location_rows = []
eval_rows = []
image_rows = []

for img_id, img in images.items():
    gt = img.get('ground_truth', {})
    gt_entities = gt.get('entities', [])
    gt_locations = gt.get('locations', [])
    
    # Image-level info
    image_rows.append({
        'image_id': int(img_id),
        'filename': img['filename'],
        'difficulty_with_gps': gt.get('difficulty_with_gps'),
        'difficulty_without_gps': gt.get('difficulty_without_gps'),
        'categories': gt.get('image_categories', []),
        'num_gt_entities': len(gt_entities),
        'num_gt_locations': len(gt_locations),
        'evaluator_notes': img.get('evaluator_notes', ''),
        'num_high_importance': sum(1 for e in gt_entities if e.get('importance') == 'high'),
        'num_medium_importance': sum(1 for e in gt_entities if e.get('importance') == 'medium'),
        'num_low_importance': sum(1 for e in gt_entities if e.get('importance') == 'low'),
        'mean_max_specificity': np.mean([e['max_specificity'] for e in gt_entities]) if gt_entities else None,
    })
    
    for ev in img['evals']:
        approach = ev['approach_name']
        use_gps = ev['use_gps']
        is_gcv = approach == 'Google Cloud Vision'
        
        # ── Process entity claims ──
        for ci, ec in enumerate(ev.get('entity_claims', [])):
            if ec.get('not_applicable', False):
                continue  # Skip N/A tagged claims
            
            gt_idxs = ec.get('gt_indices', [])
            from_loc = ec.get('from_location_claim', False)
            is_scene = ec.get('scene_description', False)
            is_bonus = ec.get('bonus', False)
            
            # Compute score
            if gt_idxs and not is_scene and not is_bonus:
                # Linked to GT: score per GT entity, pick best
                scores_per_gt = []
                for gi in gt_idxs:
                    if gi < len(gt_entities):
                        gte = gt_entities[gi]
                        s = entity_score(
                            ec.get('correctness'), 
                            ec.get('specificity'),
                            gte.get('max_specificity', 3),
                            gte.get('importance', 'medium')
                        )
                        scores_per_gt.append(s)
                claim_score = max(scores_per_gt) if scores_per_gt else entity_score_unlinked(ec.get('correctness'), ec.get('specificity'))
            else:
                claim_score = entity_score_unlinked(ec.get('correctness'), ec.get('specificity'))
            
            # Determine linked GT info
            linked_importance = None
            linked_type = None
            linked_max_spec = None
            if gt_idxs:
                first_gt = gt_entities[gt_idxs[0]] if gt_idxs[0] < len(gt_entities) else None
                if first_gt:
                    linked_importance = first_gt.get('importance')
                    linked_type = first_gt.get('type')
                    linked_max_spec = first_gt.get('max_specificity')
            
            entity_rows.append({
                'image_id': int(img_id),
                'approach_name': approach,
                'use_gps': use_gps,
                'claim_index': ci,
                'model_index': ec.get('model_index'),
                'confidence': ec.get('confidence'),
                'specificity': ec.get('specificity'),
                'correctness': ec.get('correctness'),
                'scene_description': is_scene,
                'bonus': is_bonus,
                'from_location_claim': from_loc,
                'gt_indices': gt_idxs,
                'num_gt_links': len(gt_idxs),
                'entity_score': claim_score,
                'linked_importance': linked_importance,
                'linked_type': linked_type,
                'linked_max_specificity': linked_max_spec,
                'flag': ec.get('flag', False),
            })
        
        # ── Process location claims ──
        for li, lc in enumerate(ev.get('location_claims', [])):
            loc_s = location_score(lc.get('correctness'), lc.get('specificity'))
            
            location_rows.append({
                'image_id': int(img_id),
                'approach_name': approach,
                'use_gps': use_gps,
                'claim_index': li,
                'model_index': lc.get('model_index'),
                'source': lc.get('source'),
                'gt_ref': lc.get('gt_ref'),
                'confidence': lc.get('confidence'),
                'specificity': lc.get('specificity'),
                'correctness': lc.get('correctness'),
                'location_score': loc_s,
                'also_entity': lc.get('also_entity', False),
                'flag': lc.get('flag', False),
            })

# ── Build DataFrames ──
df_entities = pd.DataFrame(entity_rows)
df_locations = pd.DataFrame(location_rows)
df_images = pd.DataFrame(image_rows)

print(f"Entity claims (excl. N/A): {len(df_entities):,}")
print(f"Location claims: {len(df_locations):,}")
print(f"Images: {len(df_images)}")


In [ ]:
# ── Aggregate to evaluation level ──
eval_agg_rows = []

for img_id, img in images.items():
    gt = img.get('ground_truth', {})
    gt_entities = gt.get('entities', [])
    
    for ev in img['evals']:
        approach = ev['approach_name']
        use_gps = ev['use_gps']
        is_gcv = approach == 'Google Cloud Vision'
        
        # ── Entity metrics ──
        e_claims = [ec for ec in ev.get('entity_claims', []) if not ec.get('not_applicable', False)]
        e_scores = []
        
        # Track which GT entities are covered by TP claims
        gt_covered = {}  # gt_index -> best score
        tp_count = 0
        fp_count = 0
        
        for ec in e_claims:
            is_scene = ec.get('scene_description', False)
            is_bonus = ec.get('bonus', False)
            gt_idxs = ec.get('gt_indices', [])
            corr = ec.get('correctness')
            spec = ec.get('specificity')
            conf = ec.get('confidence')
            
            # Score this claim
            if gt_idxs and not is_scene and not is_bonus:
                for gi in gt_idxs:
                    if gi < len(gt_entities):
                        gte = gt_entities[gi]
                        s = entity_score(corr, spec, gte.get('max_specificity', 3), gte.get('importance', 'medium'))
                        if gi not in gt_covered or s > gt_covered[gi]:
                            gt_covered[gi] = s
                # Use best GT link for the claim's score in average
                best_s = max(
                    entity_score(corr, spec, gt_entities[gi].get('max_specificity', 3), gt_entities[gi].get('importance', 'medium'))
                    for gi in gt_idxs if gi < len(gt_entities)
                ) if gt_idxs else entity_score_unlinked(corr, spec)
                e_scores.append(best_s)
            else:
                s = entity_score_unlinked(corr, spec)
                e_scores.append(s)
            
            # TP/FP counting (exclude scene and bonus from TP/FP)
            if not is_scene and not is_bonus:
                if corr is not None and corr >= 1.0 and gt_idxs:
                    max_spec = max(
                        (gt_entities[gi].get('max_specificity', 3) for gi in gt_idxs if gi < len(gt_entities)),
                        default=3
                    )
                    if spec is not None and spec >= max_spec - 1:
                        tp_count += 1
                elif corr is not None and corr < 0:
                    fp_count += 1
        
        # FN: high/medium importance GT entities not covered by any TP
        fn_count = 0
        for gi, gte in enumerate(gt_entities):
            if gte.get('added_by') == 'human_evaluator' and gte.get('importance') != 'high':
                continue  # Don't penalize for evaluator-added non-high entities
            imp = gte.get('importance', 'medium')
            max_s = gte.get('max_specificity', 1)
            if imp in ('high', 'medium') and max_s >= 3:
                # Check if covered by a TP claim
                covered = False
                for ec in e_claims:
                    if gi in ec.get('gt_indices', []):
                        corr = ec.get('correctness')
                        spec = ec.get('specificity')
                        if corr is not None and corr >= 1.0 and spec is not None and spec >= max_s - 1:
                            covered = True
                            break
                if not covered:
                    fn_count += 1
        
        entity_mean = np.mean(e_scores) if e_scores else None
        
        # ── Location metrics ──
        l_claims = ev.get('location_claims', [])
        
        # Photographer position: pick best correct claim
        photo_claims = [lc for lc in l_claims if lc.get('gt_ref') in ('photographer_position', 'both')]
        photo_score = 0.0
        if photo_claims:
            correct_photo = [lc for lc in photo_claims if lc.get('correctness', 0) > 0]
            if correct_photo:
                best = max(correct_photo, key=lambda c: (c.get('correctness', 0), c.get('specificity', 0)))
                photo_score = location_score(best['correctness'], best.get('specificity', 1))
        
        # Depicted subject: average all claims
        depicted_claims = [lc for lc in l_claims if lc.get('gt_ref') in ('depicted_subject', 'both')]
        depicted_score = 0.0
        if depicted_claims:
            depicted_scores = [location_score(lc.get('correctness'), lc.get('specificity')) for lc in depicted_claims]
            depicted_score = np.mean(depicted_scores)
        
        # Unlinked location claims (gt_ref is null)
        unlinked_loc = [lc for lc in l_claims if lc.get('gt_ref') is None]
        
        # Combined location score
        loc_components = []
        if photo_claims:
            loc_components.append(photo_score)
        if depicted_claims:
            loc_components.append(depicted_score)
        location_mean = np.mean(loc_components) if loc_components else None
        
        # Location TP/FP/FN
        loc_tp = sum(1 for lc in l_claims if lc.get('correctness', 0) >= 1.0 and lc.get('specificity', 0) >= 3 and lc.get('gt_ref') is not None)
        loc_fp = sum(1 for lc in l_claims if lc.get('correctness', 0) < 0)
        
        # Claim consistency: conflicting claims at same confidence
        conf_groups = defaultdict(list)
        for lc in l_claims:
            c = lc.get('confidence')
            if c is not None:
                conf_groups[round(c, 2)].append(lc.get('correctness', 0))
        conflicting = sum(1 for g in conf_groups.values() if len(g) > 1 and min(g) < 0 and max(g) > 0)
        
        # ── Description score ──
        desc = ev.get('desc_score')
        
        # ── Overall score ──
        ov = overall_score(entity_mean, location_mean, desc, is_gcv)
        
        eval_agg_rows.append({
            'image_id': int(img_id),
            'approach_name': approach,
            'use_gps': use_gps,
            'is_gcv': is_gcv,
            'entity_score_mean': entity_mean,
            'location_score_mean': location_mean,
            'photo_position_score': photo_score,
            'depicted_subject_score': depicted_score,
            'desc_score': desc,
            'overall_score': ov,
            'entity_tp': tp_count,
            'entity_fp': fp_count,
            'entity_fn': fn_count,
            'location_tp': loc_tp,
            'location_fp': loc_fp,
            'num_entity_claims': len(e_claims),
            'num_location_claims': len(l_claims),
            'conflicting_claims': conflicting,
        })

df_evals = pd.DataFrame(eval_agg_rows)

# Compute precision, recall, F1
df_evals['entity_precision'] = df_evals['entity_tp'] / (df_evals['entity_tp'] + df_evals['entity_fp']).replace(0, np.nan)
df_evals['entity_recall'] = df_evals['entity_tp'] / (df_evals['entity_tp'] + df_evals['entity_fn']).replace(0, np.nan)
df_evals['entity_f1'] = 2 * (df_evals['entity_precision'] * df_evals['entity_recall']) / (df_evals['entity_precision'] + df_evals['entity_recall']).replace(0, np.nan)

print(f"Evaluations: {len(df_evals):,}")
print(f"Expected: {150 * 13} = 1,950")
print(f"\nPer approach counts:")
print(df_evals.groupby(['approach_name', 'use_gps']).size().to_string())


In [ ]:
# ── Process timing data ──
# V1 batch has: GPT Vision Direct, GPT Vision + Search, Google Cloud Vision, Google Maps Places
# Mapped to our evaluation approach names

timing_map = {
    'GPT Vision Direct': 'GPT Vision Direct',
    'GPT Vision + Search': 'GPT Vision + Search', 
    'Google Cloud Vision': 'Google Cloud Vision',
}

timing_rows = []
for r in timing_raw['results']:
    fname = r['image_filename']
    use_meta = r['use_metadata']
    total_time = r.get('processing_time_seconds')
    
    for a in r.get('approaches', []):
        aname = a.get('approach')
        if aname in timing_map and a.get('success', False):
            per_approach_time = a.get('details', {}).get('processing_time_seconds')
            timing_rows.append({
                'filename': fname,
                'use_metadata': use_meta,
                'approach_base': timing_map[aname],
                'processing_time': per_approach_time,
                'total_batch_time': total_time,
            })

df_timing = pd.DataFrame(timing_rows)
print(f"Timing records: {len(df_timing):,}")
print(f"\nBy approach:")
print(df_timing.groupby('approach_base')['processing_time'].describe().round(2))


---
## Section 1: Data Overview
Dataset statistics, distributions, and quality checks.

In [ ]:
# ── Dataset Summary ──
print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Images evaluated: {len(df_images)}")
print(f"Total evaluations: {len(df_evals):,}")
print(f"Entity claims (excl. N/A): {len(df_entities):,}")
print(f"Location claims: {len(df_locations):,}")
print(f"Approaches: {df_evals['approach_name'].nunique()}")
print(f"GPS conditions: {df_evals['use_gps'].nunique()}")

print(f"\n── Difficulty Distribution ──")
for col in ['difficulty_with_gps', 'difficulty_without_gps']:
    print(f"\n{col}:")
    print(df_images[col].value_counts().to_string())

print(f"\n── Image Categories (multi-label) ──")
cat_counts = Counter()
for cats in df_images['categories']:
    for c in cats:
        cat_counts[c] += 1
for c, n in cat_counts.most_common():
    print(f"  {c}: {n}")

print(f"\n── GT Entity Importance ──")
print(f"  High:   {df_images['num_high_importance'].sum()}")
print(f"  Medium: {df_images['num_medium_importance'].sum()}")
print(f"  Low:    {df_images['num_low_importance'].sum()}")


In [ ]:
# ── Distribution plots ──
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Difficulty distributions
for i, col in enumerate(['difficulty_with_gps', 'difficulty_without_gps']):
    order = ['easy', 'medium', 'hard']
    vals = df_images[col].value_counts().reindex(order).fillna(0)
    axes[0, i].bar(vals.index, vals.values, color=['#2ca02c', '#ff7f0e', '#d62728'])
    axes[0, i].set_title(col.replace('_', ' ').title())
    axes[0, i].set_ylabel('Count')

# Category distribution
cats_series = pd.Series(cat_counts).sort_values(ascending=True)
axes[0, 2].barh(cats_series.index, cats_series.values, color='#1f77b4')
axes[0, 2].set_title('Image Categories')

# GT entities per image
axes[1, 0].hist(df_images['num_gt_entities'], bins=range(0, df_images['num_gt_entities'].max()+2), 
                color='#1f77b4', edgecolor='white')
axes[1, 0].set_title('GT Entities per Image')
axes[1, 0].set_xlabel('Count')

# Entity type distribution
type_counts = df_entities[df_entities['linked_type'].notna()]['linked_type'].value_counts()
top_types = type_counts.head(10)
axes[1, 1].barh(top_types.index[::-1], top_types.values[::-1], color='#2ca02c')
axes[1, 1].set_title('Top 10 Entity Types (linked)')

# Max specificity distribution
ms = df_images['mean_max_specificity'].dropna()
axes[1, 2].hist(ms, bins=20, color='#ff7f0e', edgecolor='white')
axes[1, 2].set_title('Mean Max Specificity per Image')
axes[1, 2].set_xlabel('Max Specificity')

plt.tight_layout()
plt.show()


In [ ]:
# ── Data quality checks ──
print("── Data Quality ──")
print(f"Null confidence in entity claims: {df_entities['confidence'].isna().sum()} / {len(df_entities)}")
print(f"Null confidence in location claims: {df_locations['confidence'].isna().sum()} / {len(df_locations)}")
print(f"Null correctness in entity claims: {df_entities['correctness'].isna().sum()}")
print(f"Null specificity in entity claims: {df_entities['specificity'].isna().sum()}")

print(f"\n── Tag distributions (entity claims) ──")
print(f"Scene description: {df_entities['scene_description'].sum()}")
print(f"Bonus: {df_entities['bonus'].sum()}")
print(f"From location claim: {df_entities['from_location_claim'].sum()}")
print(f"Flagged: {df_entities['flag'].sum()}")

print(f"\n── Location claim sources ──")
print(df_locations['source'].value_counts().to_string())
print(f"\n── Location gt_ref ──")
print(df_locations['gt_ref'].value_counts(dropna=False).to_string())

print(f"\n── Correctness distribution (entities) ──")
print(df_entities['correctness'].value_counts().sort_index().to_string())
print(f"\n── Correctness distribution (locations) ──")
print(df_locations['correctness'].value_counts().sort_index().to_string())


---
## Section 2: Overall Performance Comparison
Mean scores, statistical comparisons, and ranking.

In [ ]:
# ── Overall scores by approach × GPS ──
def approach_gps_label(row):
    short = APPROACH_SHORT[row['approach_name']]
    if row['approach_name'] == 'Google Cloud Vision':
        return short
    return f"{short} {'GPS' if row['use_gps'] else 'No GPS'}"

df_evals['approach_gps'] = df_evals.apply(approach_gps_label, axis=1)

# Summary table
summary = df_evals.groupby(['approach_name', 'use_gps']).agg(
    overall_mean=('overall_score', 'mean'),
    overall_median=('overall_score', 'median'),
    overall_std=('overall_score', 'std'),
    entity_mean=('entity_score_mean', 'mean'),
    location_mean=('location_score_mean', 'mean'),
    desc_mean=('desc_score', 'mean'),
    n=('overall_score', 'count'),
).round(4)

print("── Overall Performance Summary ──")
print(summary.to_string())


In [ ]:
# ── Box plots of overall scores ──
fig, ax = plt.subplots(figsize=(14, 7))

# Create ordered groups
groups = []
labels = []
colors = []
for approach in APPROACH_ORDER:
    if approach == 'Google Cloud Vision':
        mask = (df_evals['approach_name'] == approach)
        groups.append(df_evals.loc[mask, 'overall_score'].values)
        labels.append('GCV')
        colors.append(APPROACH_COLORS[approach])
    else:
        for gps in [True, False]:
            mask = (df_evals['approach_name'] == approach) & (df_evals['use_gps'] == gps)
            groups.append(df_evals.loc[mask, 'overall_score'].values)
            labels.append(f"{APPROACH_SHORT[approach]}\n{'GPS' if gps else 'No GPS'}")
            colors.append(APPROACH_COLORS[approach])

bp = ax.boxplot(groups, patch_artist=True, labels=labels, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Overall Score')
ax.set_title('Overall Performance by Approach and GPS Condition')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# ── Bar chart with 95% CI ──
fig, ax = plt.subplots(figsize=(14, 7))

means = []
cis = []
xlabels = []
xcolors = []

for approach in APPROACH_ORDER:
    gps_list = [False] if approach == 'Google Cloud Vision' else [True, False]
    for gps in gps_list:
        mask = (df_evals['approach_name'] == approach) & (df_evals['use_gps'] == gps)
        vals = df_evals.loc[mask, 'overall_score'].dropna()
        m = vals.mean()
        ci = 1.96 * vals.std() / np.sqrt(len(vals))
        means.append(m)
        cis.append(ci)
        if approach == 'Google Cloud Vision':
            xlabels.append('GCV')
        else:
            xlabels.append(f"{APPROACH_SHORT[approach]}\n{'GPS' if gps else 'No GPS'}")
        xcolors.append(APPROACH_COLORS[approach])

x = np.arange(len(means))
bars = ax.bar(x, means, yerr=cis, capsize=4, color=xcolors, alpha=0.8, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(xlabels, rotation=45, ha='right')
ax.set_ylabel('Mean Overall Score')
ax.set_title('Mean Overall Score with 95% CI')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

# Add value labels
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{m:.3f}', 
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── Friedman test across all 13 approach×GPS combos ──
# Need same images for all approaches - pivot to wide format
pivot = df_evals.pivot_table(index='image_id', columns='approach_gps', values='overall_score')
pivot = pivot.dropna()  # only images with all 13 evaluations

print(f"Images with all 13 evaluations: {len(pivot)}")

# Friedman test
if len(pivot) > 0 and pivot.shape[1] >= 3:
    stat, p = friedmanchisquare(*[pivot[col].values for col in pivot.columns])
    print(f"\nFriedman test: χ² = {stat:.2f}, p = {p:.2e}")
    if p < ALPHA:
        print("→ Significant differences exist among approaches (p < 0.05)")
    else:
        print("→ No significant differences found")
    
    # Post-hoc pairwise Wilcoxon with Bonferroni correction
    cols = sorted(pivot.columns)
    n_comparisons = len(cols) * (len(cols) - 1) // 2
    print(f"\nPost-hoc Wilcoxon signed-rank tests ({n_comparisons} comparisons, Bonferroni α = {ALPHA/n_comparisons:.5f}):")
    
    sig_pairs = []
    for c1, c2 in combinations(cols, 2):
        try:
            stat_w, p_w = wilcoxon(pivot[c1], pivot[c2])
            p_adj = min(p_w * n_comparisons, 1.0)
            if p_adj < ALPHA:
                d1 = pivot[c1].mean()
                d2 = pivot[c2].mean()
                sig_pairs.append((c1, c2, p_adj, d1 - d2))
        except Exception:
            pass
    
    print(f"\nSignificant pairs: {len(sig_pairs)} / {n_comparisons}")
    if sig_pairs:
        sig_pairs.sort(key=lambda x: abs(x[3]), reverse=True)
        print("\nTop 10 significant differences (by effect size):")
        for c1, c2, p_adj, diff in sig_pairs[:10]:
            print(f"  {c1} vs {c2}: Δ = {diff:+.4f}, p_adj = {p_adj:.4f}")


In [ ]:
# ── Heatmap of mean scores by component ──
components = df_evals.groupby(['approach_name', 'use_gps']).agg(
    Entity=('entity_score_mean', 'mean'),
    Location=('location_score_mean', 'mean'),
    Description=('desc_score', lambda x: ((x.dropna() - 1) / 4).mean() if x.notna().any() else np.nan),
    Overall=('overall_score', 'mean'),
).round(3)

fig, ax = plt.subplots(figsize=(10, 8))
# Build label list
labels = []
for idx in components.index:
    a, g = idx
    short = APPROACH_SHORT[a]
    labels.append(f"{short} {'GPS' if g else 'No GPS'}" if a != 'Google Cloud Vision' else 'GCV')

hm_data = components.copy()
hm_data.index = labels
sns.heatmap(hm_data, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Score'})
ax.set_title('Score Components by Approach')
plt.tight_layout()
plt.show()


---
## Section 3: Confidence Threshold Optimization
Finding optimal confidence thresholds for entity detection.

In [ ]:
# ── Sweep confidence thresholds ──
thresholds = np.arange(0.0, 1.01, 0.05)

# Only use non-scene, non-bonus entity claims for threshold analysis
mask_regular = (~df_entities['scene_description']) & (~df_entities['bonus'])
df_reg = df_entities[mask_regular].copy()

threshold_results = []

for approach in APPROACH_ORDER:
    for gps in [True, False]:
        if approach == 'Google Cloud Vision' and gps:
            continue
        
        mask = (df_reg['approach_name'] == approach) & (df_reg['use_gps'] == gps)
        claims = df_reg[mask]
        
        for thresh in thresholds:
            above = claims[claims['confidence'] >= thresh] if claims['confidence'].notna().any() else claims
            below_or_null = claims[(claims['confidence'] < thresh) | claims['confidence'].isna()]
            
            # TP: correct, linked, above threshold
            tp = above[(above['correctness'] >= 1.0) & (above['num_gt_links'] > 0)].shape[0]
            # FP: incorrect, above threshold
            fp = above[above['correctness'] < 0].shape[0]
            # FN: correct but below threshold + missed entirely
            fn_below = below_or_null[(below_or_null['correctness'] >= 1.0) & (below_or_null['num_gt_links'] > 0)].shape[0]
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn_below) if (tp + fn_below) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            mean_score = above['entity_score'].mean() if len(above) > 0 else 0
            
            threshold_results.append({
                'approach': approach,
                'use_gps': gps,
                'threshold': thresh,
                'tp': tp, 'fp': fp, 'fn': fn_below,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'mean_score': mean_score,
                'combined': f1 + 0.5 * mean_score,
                'n_above': len(above),
            })

df_thresh = pd.DataFrame(threshold_results)

# Find optimal thresholds
print("── Optimal Thresholds (max F1 + 0.5 × mean_score) ──")
for approach in APPROACH_ORDER:
    gps_list = [False] if approach == 'Google Cloud Vision' else [True, False]
    for gps in gps_list:
        mask = (df_thresh['approach'] == approach) & (df_thresh['use_gps'] == gps)
        sub = df_thresh[mask]
        if len(sub) > 0:
            best = sub.loc[sub['combined'].idxmax()]
            label = f"{APPROACH_SHORT[approach]} {'GPS' if gps else 'No GPS'}" if approach != 'Google Cloud Vision' else 'GCV'
            print(f"  {label:20s}: thresh={best['threshold']:.2f}, F1={best['f1']:.3f}, score={best['mean_score']:.3f}")


In [ ]:
# ── Precision-Recall-F1 vs threshold curves ──
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

plot_idx = 0
for approach in APPROACH_ORDER:
    gps_list = [False] if approach == 'Google Cloud Vision' else [True, False]
    for gps in gps_list:
        if plot_idx >= len(axes):
            break
        ax = axes[plot_idx]
        mask = (df_thresh['approach'] == approach) & (df_thresh['use_gps'] == gps)
        sub = df_thresh[mask].sort_values('threshold')
        
        ax.plot(sub['threshold'], sub['precision'], 'b-', label='Precision')
        ax.plot(sub['threshold'], sub['recall'], 'r-', label='Recall')
        ax.plot(sub['threshold'], sub['f1'], 'g-', linewidth=2, label='F1')
        
        # Mark optimal
        best = sub.loc[sub['combined'].idxmax()]
        ax.axvline(best['threshold'], color='gray', linestyle=':', alpha=0.5)
        
        label = f"{APPROACH_SHORT[approach]} {'GPS' if gps else 'No GPS'}" if approach != 'Google Cloud Vision' else 'GCV'
        ax.set_title(label, fontsize=10)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.05)
        if plot_idx == 0:
            ax.legend(fontsize=7)
        plot_idx += 1

# Hide unused axes
for i in range(plot_idx, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Precision / Recall / F1 vs. Confidence Threshold', fontsize=14)
fig.supxlabel('Confidence Threshold')
plt.tight_layout()
plt.show()


---
## Section 4: Precision, Recall, and F1 Analysis
Entity detection metrics stratified by approach, GPS, and difficulty.

In [ ]:
# ── Aggregate P/R/F1 per approach ──
prf = df_evals.groupby(['approach_name', 'use_gps']).agg(
    total_tp=('entity_tp', 'sum'),
    total_fp=('entity_fp', 'sum'),
    total_fn=('entity_fn', 'sum'),
    mean_precision=('entity_precision', 'mean'),
    mean_recall=('entity_recall', 'mean'),
    mean_f1=('entity_f1', 'mean'),
).round(3)

prf['macro_precision'] = prf['total_tp'] / (prf['total_tp'] + prf['total_fp'])
prf['macro_recall'] = prf['total_tp'] / (prf['total_tp'] + prf['total_fn'])
prf['macro_f1'] = 2 * prf['macro_precision'] * prf['macro_recall'] / (prf['macro_precision'] + prf['macro_recall'])

print("── Entity Detection: Precision / Recall / F1 ──")
print(prf[['total_tp', 'total_fp', 'total_fn', 'macro_precision', 'macro_recall', 'macro_f1']].round(3).to_string())


In [ ]:
# ── Grouped bar chart: P/R/F1 ──
fig, ax = plt.subplots(figsize=(16, 7))

approaches_gps = []
for a in APPROACH_ORDER:
    gps_list = [False] if a == 'Google Cloud Vision' else [True, False]
    for g in gps_list:
        short = APPROACH_SHORT[a]
        lbl = f"{short} {'GPS' if g else 'No'}" if a != 'Google Cloud Vision' else 'GCV'
        approaches_gps.append((a, g, lbl))

x = np.arange(len(approaches_gps))
width = 0.25

p_vals = [prf.loc[(a, g), 'macro_precision'] if (a, g) in prf.index else 0 for a, g, _ in approaches_gps]
r_vals = [prf.loc[(a, g), 'macro_recall'] if (a, g) in prf.index else 0 for a, g, _ in approaches_gps]
f_vals = [prf.loc[(a, g), 'macro_f1'] if (a, g) in prf.index else 0 for a, g, _ in approaches_gps]

ax.bar(x - width, p_vals, width, label='Precision', color='#1f77b4', alpha=0.8)
ax.bar(x, r_vals, width, label='Recall', color='#ff7f0e', alpha=0.8)
ax.bar(x + width, f_vals, width, label='F1', color='#2ca02c', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([lbl for _, _, lbl in approaches_gps], rotation=45, ha='right')
ax.set_ylabel('Score')
ax.set_title('Entity Detection: Precision, Recall, F1 (Macro)')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
# ── P/R/F1 stratified by difficulty ──
df_evals_diff = df_evals.merge(df_images[['image_id', 'difficulty_with_gps', 'difficulty_without_gps']], on='image_id')
df_evals_diff['difficulty'] = df_evals_diff.apply(
    lambda r: r['difficulty_with_gps'] if r['use_gps'] else r['difficulty_without_gps'], axis=1)

diff_prf = df_evals_diff.groupby(['approach_name', 'use_gps', 'difficulty']).agg(
    tp=('entity_tp', 'sum'),
    fp=('entity_fp', 'sum'),
    fn=('entity_fn', 'sum'),
)
diff_prf['precision'] = diff_prf['tp'] / (diff_prf['tp'] + diff_prf['fp'])
diff_prf['recall'] = diff_prf['tp'] / (diff_prf['tp'] + diff_prf['fn'])
diff_prf['f1'] = 2 * diff_prf['precision'] * diff_prf['recall'] / (diff_prf['precision'] + diff_prf['recall'])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, diff in enumerate(['easy', 'medium', 'hard']):
    ax = axes[i]
    sub = diff_prf.xs(diff, level='difficulty') if diff in diff_prf.index.get_level_values('difficulty') else pd.DataFrame()
    if len(sub) == 0:
        ax.set_title(f'{diff.title()} - No data')
        continue
    
    vals = []
    lbls = []
    for a in APPROACH_ORDER:
        gps_list = [False] if a == 'Google Cloud Vision' else [True, False]
        for g in gps_list:
            if (a, g) in sub.index:
                vals.append(sub.loc[(a, g), 'f1'])
                short = APPROACH_SHORT[a]
                lbls.append(f"{short}\n{'G' if g else 'N'}" if a != 'Google Cloud Vision' else 'GCV')
    
    bars = ax.bar(range(len(vals)), vals, color=[APPROACH_COLORS[a] for a in APPROACH_ORDER for g in ([False] if a == 'Google Cloud Vision' else [True, False])][:len(vals)], alpha=0.8)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(lbls, fontsize=7, rotation=45, ha='right')
    ax.set_title(f'F1 Score — {diff.title()} Images')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()


---
## Section 5: GPS Impact Analysis
Paired comparison of same approach, same image, GPS on vs. off.

In [ ]:
# ── Paired GPS comparison ──
# Exclude GCV (no GPS variant)
non_gcv = df_evals[~df_evals['is_gcv']].copy()

gps_on = non_gcv[non_gcv['use_gps'] == True].set_index(['image_id', 'approach_name'])['overall_score']
gps_off = non_gcv[non_gcv['use_gps'] == False].set_index(['image_id', 'approach_name'])['overall_score']

# Align
common = gps_on.index.intersection(gps_off.index)
delta = gps_on.loc[common] - gps_off.loc[common]

print(f"Paired comparisons: {len(delta)}")
print(f"\nOverall GPS effect:")
print(f"  Mean Δ (GPS on - off): {delta.mean():.4f}")
print(f"  Median Δ: {delta.median():.4f}")
print(f"  Improved: {(delta > 0.01).sum()} ({(delta > 0.01).mean()*100:.1f}%)")
print(f"  Degraded: {(delta < -0.01).sum()} ({(delta < -0.01).mean()*100:.1f}%)")
print(f"  Unchanged (±0.01): {((delta >= -0.01) & (delta <= 0.01)).sum()} ({((delta >= -0.01) & (delta <= 0.01)).mean()*100:.1f}%)")

# Wilcoxon signed-rank test
stat, p = wilcoxon(gps_on.loc[common], gps_off.loc[common])
print(f"\nWilcoxon signed-rank: W = {stat:.0f}, p = {p:.2e}")
if p < ALPHA:
    print("→ GPS has a statistically significant effect on performance")
else:
    print("→ No significant GPS effect overall")


In [ ]:
# ── GPS effect per approach ──
print("── GPS Effect by Approach ──")
gps_results = []
for approach in APPROACH_ORDER:
    if approach == 'Google Cloud Vision':
        continue
    
    on_mask = (non_gcv['approach_name'] == approach) & (non_gcv['use_gps'] == True)
    off_mask = (non_gcv['approach_name'] == approach) & (non_gcv['use_gps'] == False)
    
    on_vals = non_gcv[on_mask].set_index('image_id')['overall_score']
    off_vals = non_gcv[off_mask].set_index('image_id')['overall_score']
    common_imgs = on_vals.index.intersection(off_vals.index)
    
    if len(common_imgs) > 10:
        d = on_vals.loc[common_imgs] - off_vals.loc[common_imgs]
        stat_w, p_w = wilcoxon(on_vals.loc[common_imgs], off_vals.loc[common_imgs])
        gps_results.append({
            'approach': APPROACH_SHORT[approach],
            'mean_delta': d.mean(),
            'median_delta': d.median(),
            'pct_improved': (d > 0.01).mean() * 100,
            'pct_degraded': (d < -0.01).mean() * 100,
            'wilcoxon_p': p_w,
            'significant': p_w < ALPHA,
        })

df_gps_effect = pd.DataFrame(gps_results)
print(df_gps_effect.to_string(index=False))


In [ ]:
# ── GPS effect visualization ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution of deltas per approach
ax = axes[0]
for approach in APPROACH_ORDER:
    if approach == 'Google Cloud Vision':
        continue
    on = non_gcv[(non_gcv['approach_name'] == approach) & (non_gcv['use_gps'] == True)].set_index('image_id')['overall_score']
    off = non_gcv[(non_gcv['approach_name'] == approach) & (non_gcv['use_gps'] == False)].set_index('image_id')['overall_score']
    common_imgs = on.index.intersection(off.index)
    d = on.loc[common_imgs] - off.loc[common_imgs]
    ax.hist(d, bins=30, alpha=0.5, label=APPROACH_SHORT[approach], color=APPROACH_COLORS[approach])

ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Δ Score (GPS on − GPS off)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of GPS Effect')
ax.legend(fontsize=7)

# Per-approach mean delta with CI
ax = axes[1]
names = []
means = []
cis = []
colors_list = []
for approach in APPROACH_ORDER:
    if approach == 'Google Cloud Vision':
        continue
    on = non_gcv[(non_gcv['approach_name'] == approach) & (non_gcv['use_gps'] == True)].set_index('image_id')['overall_score']
    off = non_gcv[(non_gcv['approach_name'] == approach) & (non_gcv['use_gps'] == False)].set_index('image_id')['overall_score']
    common_imgs = on.index.intersection(off.index)
    d = on.loc[common_imgs] - off.loc[common_imgs]
    names.append(APPROACH_SHORT[approach])
    means.append(d.mean())
    cis.append(1.96 * d.std() / np.sqrt(len(d)))
    colors_list.append(APPROACH_COLORS[approach])

y = np.arange(len(names))
ax.barh(y, means, xerr=cis, color=colors_list, alpha=0.8, capsize=4)
ax.set_yticks(y)
ax.set_yticklabels(names)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Mean Δ Score (GPS on − GPS off)')
ax.set_title('GPS Effect by Approach (with 95% CI)')

plt.tight_layout()
plt.show()


In [ ]:
# ── GPS effect stratified by difficulty ──
df_delta = non_gcv.merge(df_images[['image_id', 'difficulty_with_gps', 'difficulty_without_gps']], on='image_id')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, diff in enumerate(['easy', 'medium', 'hard']):
    ax = axes[i]
    deltas_by_approach = []
    labels = []
    colors_d = []
    
    for approach in APPROACH_ORDER:
        if approach == 'Google Cloud Vision':
            continue
        on = df_delta[(df_delta['approach_name'] == approach) & (df_delta['use_gps'] == True) & 
                      (df_delta['difficulty_with_gps'] == diff)].set_index('image_id')['overall_score']
        off = df_delta[(df_delta['approach_name'] == approach) & (df_delta['use_gps'] == False) & 
                       (df_delta['difficulty_without_gps'] == diff)].set_index('image_id')['overall_score']
        common_imgs = on.index.intersection(off.index)
        if len(common_imgs) > 0:
            d = on.loc[common_imgs] - off.loc[common_imgs]
            deltas_by_approach.append(d.values)
            labels.append(APPROACH_SHORT[approach])
            colors_d.append(APPROACH_COLORS[approach])
    
    if deltas_by_approach:
        bp = ax.boxplot(deltas_by_approach, patch_artist=True, labels=labels, widths=0.6)
        for patch, color in zip(bp['boxes'], colors_d):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_title(f'GPS Effect — {diff.title()} Images')
    ax.set_ylabel('Δ Score' if i == 0 else '')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.suptitle('GPS Effect by Difficulty', fontsize=14)
plt.tight_layout()
plt.show()


---
## Section 6: Strong vs. Fast Model Comparison
Performance gain vs. processing time trade-off.

In [ ]:
# ── Strong vs Fast paired comparison ──
model_pairs = [
    ('GPT Vision Direct (strong)', 'GPT Vision Direct (fast)'),
    ('GPT Vision + Search (strong)', 'GPT Vision + Search (fast)'),
    ('Gemini Vision (strong)', 'Gemini Vision (fast)'),
]

print("── Strong vs. Fast Model Comparison ──")
sf_results = []
for strong, fast in model_pairs:
    for gps in [True, False]:
        s_vals = df_evals[(df_evals['approach_name'] == strong) & (df_evals['use_gps'] == gps)].set_index('image_id')['overall_score']
        f_vals = df_evals[(df_evals['approach_name'] == fast) & (df_evals['use_gps'] == gps)].set_index('image_id')['overall_score']
        common = s_vals.index.intersection(f_vals.index)
        
        if len(common) > 10:
            d = s_vals.loc[common] - f_vals.loc[common]
            stat_w, p_w = wilcoxon(s_vals.loc[common], f_vals.loc[common])
            
            base = strong.split('(')[0].strip()
            sf_results.append({
                'model_family': base,
                'gps': 'GPS' if gps else 'No GPS',
                'strong_mean': s_vals.loc[common].mean(),
                'fast_mean': f_vals.loc[common].mean(),
                'delta_mean': d.mean(),
                'delta_pct': d.mean() / f_vals.loc[common].mean() * 100 if f_vals.loc[common].mean() != 0 else 0,
                'wilcoxon_p': p_w,
                'significant': p_w < ALPHA,
            })

df_sf = pd.DataFrame(sf_results)
print(df_sf.to_string(index=False))


In [ ]:
# ── Strong vs. Fast scatter with timing ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Performance comparison
ax = axes[0]
for strong, fast in model_pairs:
    for gps in [True, False]:
        s = df_evals[(df_evals['approach_name'] == strong) & (df_evals['use_gps'] == gps)]['overall_score'].mean()
        f = df_evals[(df_evals['approach_name'] == fast) & (df_evals['use_gps'] == gps)]['overall_score'].mean()
        marker = 'o' if gps else 's'
        base = strong.split('(')[0].strip()
        ax.plot([f], [s], marker=marker, markersize=10, color=APPROACH_COLORS[strong],
                label=f"{base} {'GPS' if gps else 'No GPS'}")

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Equal performance')
ax.set_xlabel('Fast Model Score')
ax.set_ylabel('Strong Model Score')
ax.set_title('Strong vs. Fast: Mean Overall Score')
ax.legend(fontsize=8)

# Per-image deltas
ax = axes[1]
for strong, fast in model_pairs:
    s_vals = df_evals[(df_evals['approach_name'] == strong) & (df_evals['use_gps'] == True)].set_index('image_id')['overall_score']
    f_vals = df_evals[(df_evals['approach_name'] == fast) & (df_evals['use_gps'] == True)].set_index('image_id')['overall_score']
    common = s_vals.index.intersection(f_vals.index)
    d = s_vals.loc[common] - f_vals.loc[common]
    base = strong.split('(')[0].strip()
    ax.hist(d, bins=25, alpha=0.5, label=f"{base}", color=APPROACH_COLORS[strong])

ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Δ Score (Strong − Fast)')
ax.set_ylabel('Frequency')
ax.set_title('Score Difference Distribution (GPS on)')
ax.legend()

plt.tight_layout()
plt.show()


---
## Section 7: Entity Type Performance
Detection rates and specificity achievement by entity type.

In [ ]:
# ── Entity type analysis ──
# Only linked claims (have gt_indices)
linked = df_entities[(df_entities['num_gt_links'] > 0) & (~df_entities['scene_description']) & (~df_entities['bonus'])].copy()

type_perf = linked.groupby(['linked_type', 'approach_name', 'use_gps']).agg(
    n_claims=('entity_score', 'count'),
    mean_score=('entity_score', 'mean'),
    mean_correctness=('correctness', 'mean'),
    mean_specificity=('specificity', 'mean'),
    pct_correct=('correctness', lambda x: (x >= 1.0).mean()),
).round(3)

# Overall by type
type_overall = linked.groupby('linked_type').agg(
    n_claims=('entity_score', 'count'),
    mean_score=('entity_score', 'mean'),
    mean_correctness=('correctness', 'mean'),
    pct_fully_correct=('correctness', lambda x: (x >= 1.0).mean()),
    mean_specificity=('specificity', 'mean'),
).sort_values('n_claims', ascending=False)

print("── Entity Type Performance (Overall) ──")
print(type_overall.round(3).to_string())


In [ ]:
# ── Entity type heatmap ──
# Top entity types by frequency
top_types = type_overall.head(8).index.tolist()

pivot_type = linked[linked['linked_type'].isin(top_types)].groupby(
    ['linked_type', 'approach_name']
)['entity_score'].mean().unstack(level='approach_name')

# Reorder
pivot_type = pivot_type.reindex(columns=[a for a in APPROACH_ORDER if a in pivot_type.columns])
pivot_type.columns = [APPROACH_SHORT.get(c, c) for c in pivot_type.columns]

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot_type, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5)
ax.set_title('Mean Entity Score by Type × Approach')
ax.set_ylabel('Entity Type')
plt.tight_layout()
plt.show()


In [ ]:
# ── Kruskal-Wallis test: entity score varies by type? ──
type_groups = [g['entity_score'].values for _, g in linked.groupby('linked_type') if len(g) >= 10]
if len(type_groups) >= 3:
    stat, p = kruskal(*type_groups)
    print(f"Kruskal-Wallis test (entity score ~ type): H = {stat:.2f}, p = {p:.2e}")
    if p < ALPHA:
        print("→ Significant variation in entity scores across entity types")

# ── Landmark focus: best/worst approaches ──
landmarks = linked[linked['linked_type'] == 'landmark']
if len(landmarks) > 0:
    lm_perf = landmarks.groupby(['approach_name', 'use_gps']).agg(
        mean_score=('entity_score', 'mean'),
        pct_correct=('correctness', lambda x: (x >= 1.0).mean()),
        n=('entity_score', 'count'),
    ).sort_values('mean_score', ascending=False)
    print("\n── Landmark Detection Performance ──")
    print(lm_perf.round(3).to_string())


---
## Section 8: Important Entity False Negatives
Analysis of critical misses — high-importance entities that models failed to identify.

In [ ]:
# ── FN analysis ──
fn_data = []

for img_id, img in images.items():
    gt = img.get('ground_truth', {})
    gt_entities = gt.get('entities', [])
    
    for ev in img['evals']:
        approach = ev['approach_name']
        use_gps = ev['use_gps']
        e_claims = ev.get('entity_claims', [])
        
        for gi, gte in enumerate(gt_entities):
            imp = gte.get('importance', 'medium')
            max_s = gte.get('max_specificity', 1)
            etype = gte.get('type')
            
            if imp not in ('high', 'medium') or max_s < 3:
                continue
            
            # Check if covered by a correct claim
            covered = False
            best_corr = None
            for ec in e_claims:
                if gi in ec.get('gt_indices', []):
                    corr = ec.get('correctness')
                    spec = ec.get('specificity')
                    if corr is not None and corr >= 1.0 and spec is not None and spec >= max_s - 1:
                        covered = True
                        break
                    if best_corr is None or (corr is not None and corr > best_corr):
                        best_corr = corr
            
            if not covered:
                fn_data.append({
                    'image_id': int(img_id),
                    'approach_name': approach,
                    'use_gps': use_gps,
                    'gt_index': gi,
                    'entity_name': gte.get('name', ''),
                    'entity_type': etype,
                    'importance': imp,
                    'max_specificity': max_s,
                    'best_correctness': best_corr,
                })

df_fn = pd.DataFrame(fn_data)
print(f"Total false negative instances: {len(df_fn):,}")
print(f"Unique GT entities missed at least once: {df_fn[['image_id', 'gt_index']].drop_duplicates().shape[0]}")

# FN rate per approach
fn_rate = df_fn.groupby(['approach_name', 'use_gps']).size().reset_index(name='fn_count')
fn_rate = fn_rate.sort_values('fn_count', ascending=True)
print("\n── FN Count by Approach ──")
print(fn_rate.to_string(index=False))


In [ ]:
# ── FN by importance and approach ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# FN count bar chart
ax = axes[0]
fn_by_approach = df_fn.groupby(['approach_name', 'use_gps']).size().reset_index(name='fn_count')
fn_by_approach['label'] = fn_by_approach.apply(
    lambda r: f"{APPROACH_SHORT[r['approach_name']]}\n{'G' if r['use_gps'] else 'N'}" 
    if r['approach_name'] != 'Google Cloud Vision' else 'GCV', axis=1)
fn_by_approach = fn_by_approach.sort_values('fn_count')

bars = ax.barh(fn_by_approach['label'], fn_by_approach['fn_count'],
               color=[APPROACH_COLORS[a] for a in fn_by_approach['approach_name']], alpha=0.8)
ax.set_xlabel('False Negative Count')
ax.set_title('Total Entity False Negatives by Approach')

# FN by entity type
ax = axes[1]
fn_type = df_fn.groupby('entity_type').size().sort_values(ascending=True)
ax.barh(fn_type.index, fn_type.values, color='#d62728', alpha=0.7)
ax.set_xlabel('FN Count')
ax.set_title('False Negatives by Entity Type')

plt.tight_layout()
plt.show()


In [ ]:
# ── Heatmap: FN rate per approach × difficulty ──
df_fn_diff = df_fn.merge(df_images[['image_id', 'difficulty_with_gps', 'difficulty_without_gps']], on='image_id')
df_fn_diff['difficulty'] = df_fn_diff.apply(
    lambda r: r['difficulty_with_gps'] if r['use_gps'] else r['difficulty_without_gps'], axis=1)

fn_heat = df_fn_diff.groupby(['approach_name', 'use_gps', 'difficulty']).size().unstack(fill_value=0)
fn_heat.index = [f"{APPROACH_SHORT[a]} {'GPS' if g else 'No'}" if a != 'Google Cloud Vision' else 'GCV'
                 for a, g in fn_heat.index]

if not fn_heat.empty:
    fig, ax = plt.subplots(figsize=(8, 8))
    col_order = [c for c in ['easy', 'medium', 'hard'] if c in fn_heat.columns]
    sns.heatmap(fn_heat[col_order], annot=True, fmt='d', cmap='Reds', ax=ax, linewidths=0.5)
    ax.set_title('False Negative Count by Approach × Difficulty')
    plt.tight_layout()
    plt.show()


---
## Section 9: Description Score Analysis
Quality of free-text scene descriptions (excluding GCV).

In [ ]:
# ── Description score distribution ──
desc_data = df_evals[df_evals['desc_score'].notna()].copy()

desc_summary = desc_data.groupby(['approach_name', 'use_gps']).agg(
    mean_desc=('desc_score', 'mean'),
    median_desc=('desc_score', 'median'),
    std_desc=('desc_score', 'std'),
).round(3)

print("── Description Score Summary ──")
print(desc_summary.to_string())

# Correlation with entity and location scores
print("\n── Correlations ──")
for col in ['entity_score_mean', 'location_score_mean']:
    valid = desc_data[[col, 'desc_score']].dropna()
    if len(valid) > 10:
        rho, p = spearmanr(valid[col], valid['desc_score'])
        print(f"  desc_score vs {col}: ρ = {rho:.3f}, p = {p:.2e}")


In [ ]:
# ── Description score bar chart ──
fig, ax = plt.subplots(figsize=(14, 6))

labels = []
means = []
colors_list = []
for a in APPROACH_ORDER:
    if a == 'Google Cloud Vision':
        continue
    for g in [True, False]:
        mask = (desc_data['approach_name'] == a) & (desc_data['use_gps'] == g)
        vals = desc_data.loc[mask, 'desc_score']
        if len(vals) > 0:
            labels.append(f"{APPROACH_SHORT[a]}\n{'GPS' if g else 'No GPS'}")
            means.append(vals.mean())
            colors_list.append(APPROACH_COLORS[a])

x = np.arange(len(labels))
bars = ax.bar(x, means, color=colors_list, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Mean Description Score (1-5)')
ax.set_title('Description Quality by Approach')
ax.set_ylim(1, 5)

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{m:.2f}',
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


---
## Section 10: Confidence Calibration
How well does model confidence predict actual correctness?

In [ ]:
# ── Calibration analysis ──
# Entity claims with valid confidence
cal_data = df_entities[df_entities['confidence'].notna() & df_entities['correctness'].notna()].copy()
cal_data['correct_binary'] = (cal_data['correctness'] >= 1.0).astype(int)

def compute_ece(conf, correct, n_bins=10):
    """Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (conf >= bins[i]) & (conf < bins[i+1])
        if mask.sum() > 0:
            avg_conf = conf[mask].mean()
            avg_acc = correct[mask].mean()
            ece += mask.sum() / len(conf) * abs(avg_conf - avg_acc)
    return ece

print("── Confidence Calibration (ECE) ──")
cal_results = []
for approach in APPROACH_ORDER:
    for gps in [True, False]:
        if approach == 'Google Cloud Vision' and gps:
            continue
        mask = (cal_data['approach_name'] == approach) & (cal_data['use_gps'] == gps)
        sub = cal_data[mask]
        if len(sub) > 20:
            ece = compute_ece(sub['confidence'].values, sub['correct_binary'].values)
            rho, p = spearmanr(sub['confidence'], sub['correctness'])
            over_conf = ((sub['confidence'] > 0.7) & (sub['correctness'] < 0)).mean()
            
            label = f"{APPROACH_SHORT[approach]} {'GPS' if gps else 'No GPS'}" if approach != 'Google Cloud Vision' else 'GCV'
            cal_results.append({
                'approach': label,
                'ECE': ece,
                'conf_corr_rho': rho,
                'conf_corr_p': p,
                'overconfidence_rate': over_conf,
                'mean_confidence': sub['confidence'].mean(),
                'n': len(sub),
            })

df_cal = pd.DataFrame(cal_results)
print(df_cal.round(4).to_string(index=False))


In [ ]:
# ── Calibration plots ──
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
n_bins = 10

plot_idx = 0
for approach in APPROACH_ORDER:
    gps_list = [False] if approach == 'Google Cloud Vision' else [True, False]
    for gps in gps_list:
        if plot_idx >= len(axes):
            break
        ax = axes[plot_idx]
        mask = (cal_data['approach_name'] == approach) & (cal_data['use_gps'] == gps)
        sub = cal_data[mask]
        
        if len(sub) > 20:
            bins = np.linspace(0, 1, n_bins + 1)
            bin_centers = []
            bin_accs = []
            bin_counts = []
            for i in range(n_bins):
                bmask = (sub['confidence'] >= bins[i]) & (sub['confidence'] < bins[i+1])
                if bmask.sum() > 0:
                    bin_centers.append((bins[i] + bins[i+1]) / 2)
                    bin_accs.append(sub.loc[bmask, 'correct_binary'].mean())
                    bin_counts.append(bmask.sum())
            
            ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect')
            ax.bar(bin_centers, bin_accs, width=0.08, alpha=0.7, color=APPROACH_COLORS[approach])
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1.05)
        
        label = f"{APPROACH_SHORT[approach]} {'GPS' if gps else 'No GPS'}" if approach != 'Google Cloud Vision' else 'GCV'
        ax.set_title(label, fontsize=9)
        if plot_idx % 4 == 0:
            ax.set_ylabel('Accuracy')
        if plot_idx >= 4:
            ax.set_xlabel('Confidence')
        plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Calibration: Confidence vs. Actual Accuracy (Entity Claims)', fontsize=13)
plt.tight_layout()
plt.show()


---
## Section 11: Per-Image Performance Heatmap
Visualizing all 150 images × 13 approaches.

In [ ]:
# ── Build pivot for heatmap ──
heat_pivot = df_evals.pivot_table(
    index='image_id', 
    columns='approach_gps', 
    values='overall_score'
)

# Sort by mean difficulty (proxy: mean score across approaches)
heat_pivot['mean_score'] = heat_pivot.mean(axis=1)
heat_pivot = heat_pivot.sort_values('mean_score', ascending=False)
heat_pivot = heat_pivot.drop('mean_score', axis=1)

# Reorder columns logically
col_order = []
for a in APPROACH_ORDER:
    if a == 'Google Cloud Vision':
        col_order.append('GCV')
    else:
        col_order.append(f"{APPROACH_SHORT[a]} GPS")
        col_order.append(f"{APPROACH_SHORT[a]} No GPS")

col_order = [c for c in col_order if c in heat_pivot.columns]
heat_pivot = heat_pivot[col_order]

fig, ax = plt.subplots(figsize=(16, 20))
sns.heatmap(heat_pivot, cmap='RdYlGn', center=0, ax=ax, 
            xticklabels=True, yticklabels=False,
            linewidths=0, cbar_kws={'label': 'Overall Score', 'shrink': 0.5})
ax.set_title('Per-Image Performance Heatmap (sorted by mean score)')
ax.set_xlabel('Approach')
ax.set_ylabel(f'Images (n={len(heat_pivot)})')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

print(f"Heatmap: {heat_pivot.shape[0]} images × {heat_pivot.shape[1]} approaches")
print(f"Score range: [{heat_pivot.min().min():.3f}, {heat_pivot.max().max():.3f}]")


---
## Section 12: Image Category Comparison
Performance across scene types: landmark, urban, nature, art, other.

In [ ]:
# ── Explode categories for stratified analysis ──
df_img_cat = df_images.explode('categories')
df_eval_cat = df_evals.merge(df_img_cat[['image_id', 'categories']], on='image_id')

cat_perf = df_eval_cat.groupby(['categories', 'approach_name', 'use_gps']).agg(
    mean_overall=('overall_score', 'mean'),
    mean_entity=('entity_score_mean', 'mean'),
    n=('overall_score', 'count'),
).round(3)

# Overall by category
cat_overall = df_eval_cat.groupby('categories')['overall_score'].agg(['mean', 'median', 'std', 'count']).round(3)
print("── Performance by Image Category ──")
print(cat_overall.to_string())


In [ ]:
# ── Category × Approach heatmap ──
cat_heat = df_eval_cat.groupby(['categories', 'approach_name'])['overall_score'].mean().unstack()
cat_heat = cat_heat.reindex(columns=[a for a in APPROACH_ORDER if a in cat_heat.columns])
cat_heat.columns = [APPROACH_SHORT.get(c, c) for c in cat_heat.columns]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(cat_heat, annot=True, fmt='.3f', cmap='RdYlGn', center=cat_heat.values.mean(), 
            ax=ax, linewidths=0.5)
ax.set_title('Mean Overall Score: Category × Approach')
ax.set_ylabel('Image Category')
plt.tight_layout()
plt.show()


In [ ]:
# ── Radar chart per approach family ──
categories = sorted(df_eval_cat['categories'].unique())
n_cats = len(categories)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]

for approach in APPROACH_ORDER:
    vals = []
    for cat in categories:
        mask = (df_eval_cat['categories'] == cat) & (df_eval_cat['approach_name'] == approach)
        v = df_eval_cat.loc[mask, 'overall_score'].mean()
        vals.append(v if not np.isnan(v) else 0)
    vals += vals[:1]
    
    style = '-' if 'strong' in approach or approach == 'Google Cloud Vision' else '--'
    ax.plot(angles, vals, style, linewidth=1.5, label=APPROACH_SHORT[approach], 
            color=APPROACH_COLORS[approach])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_title('Performance by Category (Radar)', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=7)
plt.tight_layout()
plt.show()


---
## Section 13: Wrong Claims Analysis
Analyzing incorrect claims, error patterns, and model reliability.

In [ ]:
# ── Wrong entity claims ──
wrong_entities = df_entities[df_entities['correctness'] < 0].copy()
print(f"Wrong entity claims (correctness < 0): {len(wrong_entities):,}")
print(f"  Of which correctness = -1.0 (clearly wrong): {(wrong_entities['correctness'] == -1.0).sum()}")
print(f"  Of which correctness = -0.5 (plausible but wrong): {(wrong_entities['correctness'] == -0.5).sum()}")

# Wrong claims by approach
wrong_by_approach = wrong_entities.groupby(['approach_name', 'use_gps']).agg(
    n_wrong=('correctness', 'count'),
    mean_confidence=('confidence', 'mean'),
    pct_high_conf=('confidence', lambda x: (x > 0.7).mean() * 100),
).round(2)

print("\n── Wrong Entity Claims by Approach ──")
print(wrong_by_approach.to_string())


In [ ]:
# ── Wrong location claims ──
wrong_locs = df_locations[df_locations['correctness'] < 0].copy()
print(f"Wrong location claims (correctness < 0): {len(wrong_locs):,}")

wrong_loc_approach = wrong_locs.groupby(['approach_name', 'use_gps']).agg(
    n_wrong=('correctness', 'count'),
    mean_confidence=('confidence', 'mean'),
    pct_high_conf=('confidence', lambda x: (x > 0.7).mean() * 100),
).round(2)

print("\n── Wrong Location Claims by Approach ──")
print(wrong_loc_approach.to_string())


In [ ]:
# ── Confidence of wrong vs correct claims ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entity confidence by correctness
ax = axes[0]
for corr_val in [1.0, 0.5, -0.5, -1.0]:
    sub = df_entities[df_entities['correctness'] == corr_val]['confidence'].dropna()
    if len(sub) > 10:
        ax.hist(sub, bins=20, alpha=0.5, label=f'Corr={corr_val}', 
                color=CORR_CMAP.get(corr_val, 'gray'))
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.set_title('Entity Claim Confidence by Correctness')
ax.legend()

# Location confidence by correctness
ax = axes[1]
for corr_val in [1.0, 0.5, -0.5, -1.0]:
    sub = df_locations[df_locations['correctness'] == corr_val]['confidence'].dropna()
    if len(sub) > 10:
        ax.hist(sub, bins=20, alpha=0.5, label=f'Corr={corr_val}',
                color=CORR_CMAP.get(corr_val, 'gray'))
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.set_title('Location Claim Confidence by Correctness')
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── Claim consistency metric ──
# How often do models make conflicting claims?
consistency = df_evals.groupby(['approach_name', 'use_gps'])['conflicting_claims'].agg(['sum', 'mean']).round(3)
consistency.columns = ['total_conflicts', 'mean_per_image']
print("── Claim Consistency (conflicting claims at same confidence) ──")
print(consistency.sort_values('total_conflicts', ascending=False).to_string())

# High-confidence wrong claims (hallucination indicator)
hall = df_entities[(df_entities['correctness'] < 0) & (df_entities['confidence'] > 0.8)]
print(f"\n── High-Confidence Wrong Claims (conf > 0.8, corr < 0) ──")
print(f"Total: {len(hall)}")
hall_by_approach = hall.groupby(['approach_name', 'use_gps']).size().sort_values(ascending=False)
print(hall_by_approach.to_string())


---
## Section 14: Processing Time vs. Performance
Efficiency analysis (timing data available for GPT Direct, GPT+Search, GCV).

In [ ]:
# ── Timing analysis ──
# Map timing approach names to evaluation approach names
# V1 batch only has: GPT Vision Direct, GPT Vision + Search, Google Cloud Vision
# These correspond to the strong variants in the evaluation

timing_summary = df_timing.groupby(['approach_base', 'use_metadata']).agg(
    mean_time=('processing_time', 'mean'),
    median_time=('processing_time', 'median'),
    std_time=('processing_time', 'std'),
    n=('processing_time', 'count'),
).round(2)

print("── Processing Time Summary (seconds) ──")
print(timing_summary.to_string())

# Efficiency: score per second
print("\n── Efficiency (Score per Second) ──")
for approach_base in ['GPT Vision Direct', 'GPT Vision + Search', 'Google Cloud Vision']:
    for use_meta in [True, False]:
        t_mask = (df_timing['approach_base'] == approach_base) & (df_timing['use_metadata'] == use_meta)
        mean_time = df_timing.loc[t_mask, 'processing_time'].mean()
        
        if mean_time > 0:
            # Match to eval approach (strong variant)
            eval_name = approach_base + ' (strong)' if approach_base != 'Google Cloud Vision' else approach_base
            gps = use_meta
            if approach_base == 'Google Cloud Vision':
                gps = False  # GCV always no GPS
            
            e_mask = (df_evals['approach_name'] == eval_name) & (df_evals['use_gps'] == gps)
            mean_score = df_evals.loc[e_mask, 'overall_score'].mean()
            
            if not np.isnan(mean_score):
                efficiency = mean_score / mean_time
                label = f"{approach_base} ({'GPS' if use_meta else 'No GPS'})"
                print(f"  {label:40s}: {mean_score:.3f} / {mean_time:.1f}s = {efficiency:.4f} score/s")


In [ ]:
# ── Scatter: time vs score ──
fig, ax = plt.subplots(figsize=(10, 7))

scatter_data = []
for approach_base in ['GPT Vision Direct', 'GPT Vision + Search', 'Google Cloud Vision']:
    for use_meta in [True, False]:
        if approach_base == 'Google Cloud Vision' and use_meta:
            continue
        t_mask = (df_timing['approach_base'] == approach_base) & (df_timing['use_metadata'] == use_meta)
        mean_time = df_timing.loc[t_mask, 'processing_time'].mean()
        
        # Map to strong eval
        eval_name = approach_base + ' (strong)' if approach_base != 'Google Cloud Vision' else approach_base
        gps = use_meta if approach_base != 'Google Cloud Vision' else False
        
        e_mask = (df_evals['approach_name'] == eval_name) & (df_evals['use_gps'] == gps)
        mean_score = df_evals.loc[e_mask, 'overall_score'].mean()
        
        if not np.isnan(mean_score) and mean_time > 0:
            marker = 'o' if use_meta else 's'
            color = APPROACH_COLORS.get(eval_name, '#333333')
            label = f"{APPROACH_SHORT.get(eval_name, approach_base)} {'GPS' if use_meta else 'No GPS'}"
            ax.scatter(mean_time, mean_score, s=150, marker=marker, color=color, label=label, zorder=3)
            scatter_data.append((mean_time, mean_score))

if scatter_data:
    # Add efficiency lines (iso-efficiency)
    times = np.linspace(0.1, max(t for t, s in scatter_data) * 1.5, 100)
    for eff in [0.01, 0.02, 0.05]:
        ax.plot(times, eff * times, ':', alpha=0.2, color='gray')

ax.set_xlabel('Mean Processing Time (seconds)')
ax.set_ylabel('Mean Overall Score')
ax.set_title('Processing Time vs. Performance')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


---
## Section 15: Google Cloud Vision Special Analysis
Entity-only comparison without description penalty.

In [ ]:
# ── GCV vs others (entity-only comparison) ──
gcv_evals = df_evals[df_evals['is_gcv']].copy()
non_gcv_evals = df_evals[~df_evals['is_gcv']].copy()

print("── GCV Performance ──")
print(f"Mean overall score: {gcv_evals['overall_score'].mean():.4f}")
print(f"Mean entity score: {gcv_evals['entity_score_mean'].mean():.4f}")
print(f"Mean location score: {gcv_evals['location_score_mean'].mean():.4f}")
print(f"Mean entity precision: {gcv_evals['entity_precision'].mean():.4f}")
print(f"Mean entity recall: {gcv_evals['entity_recall'].mean():.4f}")

# Fair comparison: entity score only (no description penalty)
print("\n── Entity-Only Score Comparison (Fair to GCV) ──")
entity_only = df_evals.groupby(['approach_name', 'use_gps'])['entity_score_mean'].mean().sort_values(ascending=False)
print(entity_only.round(4).to_string())


In [ ]:
# ── GCV entity type focus ──
gcv_entities = df_entities[df_entities['approach_name'] == 'Google Cloud Vision']

# What types does GCV detect?
gcv_type_dist = gcv_entities.groupby('linked_type').agg(
    n=('entity_score', 'count'),
    mean_score=('entity_score', 'mean'),
    pct_correct=('correctness', lambda x: (x >= 1.0).mean()),
).sort_values('n', ascending=False)

print("── GCV Entity Detection by Type ──")
print(gcv_type_dist.round(3).to_string())

# GCV scene descriptions
gcv_scenes = gcv_entities[gcv_entities['scene_description']].shape[0]
gcv_non_scenes = gcv_entities[~gcv_entities['scene_description']].shape[0]
print(f"\nGCV scene descriptions: {gcv_scenes}")
print(f"GCV specific entity claims: {gcv_non_scenes}")


---
## Section 16: Qualitative Integration
Mapping evaluator observations to quantitative patterns.

In [ ]:
# ── Evaluator notes analysis ──
notes = df_images[df_images['evaluator_notes'].str.len() > 0]['evaluator_notes']
print(f"Images with evaluator notes: {len(notes)}")

# Extract common themes
from collections import Counter
import re

themes = Counter()
theme_patterns = {
    'gemini_strength': r'[Gg]emini.*(good|strong|correct|recogniz)',
    'gemini_wrong': r'[Gg]emini.*(wrong|incorrect|confid)',
    'gpt_strength': r'[Gg][Pp][Tt].*(good|strong|correct)',
    'gpt_wrong': r'[Gg][Pp][Tt].*(wrong|incorrect|halluc)',
    'gps_helps': r'[Gg][Pp][Ss].*(help|improv|correct)',
    'gps_hurts': r'[Gg][Pp][Ss].*(wrong|mislead|hurt)',
    'difficult': r'(difficult|tricky|hard|obscur|weird angle)',
    'landmark': r'(landmark|monument|building|church|cathedral)',
    'text_ocr': r'(text|sign|read|OCR|letter)',
}

for note in notes:
    for theme, pattern in theme_patterns.items():
        if re.search(pattern, note, re.IGNORECASE):
            themes[theme] += 1

print("\n── Theme Frequency in Evaluator Notes ──")
for theme, count in themes.most_common():
    print(f"  {theme}: {count}")


In [ ]:
# ── Performance on 'tricky' images ──
tricky_mask = df_images['evaluator_notes'].str.contains(
    r'difficult|tricky|hard|obscur|weird|angle|obstruct', case=False, na=False)
tricky_ids = df_images[tricky_mask]['image_id'].values
normal_ids = df_images[~tricky_mask]['image_id'].values

print(f"'Tricky' images (from notes): {len(tricky_ids)}")
print(f"Normal images: {len(normal_ids)}")

tricky_scores = df_evals[df_evals['image_id'].isin(tricky_ids)]['overall_score']
normal_scores = df_evals[df_evals['image_id'].isin(normal_ids)]['overall_score']

print(f"\nMean score on tricky: {tricky_scores.mean():.4f}")
print(f"Mean score on normal: {normal_scores.mean():.4f}")

if len(tricky_scores) > 10 and len(normal_scores) > 10:
    stat, p = mannwhitneyu(tricky_scores, normal_scores, alternative='less')
    print(f"Mann-Whitney U: U = {stat:.0f}, p = {p:.4f}")
    if p < ALPHA:
        print("→ Significantly lower performance on 'tricky' images")


---
## Section 17: Bonus Points Analysis
Which approaches discover entities beyond the ground truth?

In [ ]:
# ── Bonus entity analysis ──
bonus = df_entities[df_entities['bonus']].copy()
print(f"Total bonus-tagged claims: {len(bonus)}")

bonus_by_approach = bonus.groupby(['approach_name', 'use_gps']).agg(
    n_bonus=('entity_score', 'count'),
    mean_score=('entity_score', 'mean'),
    mean_specificity=('specificity', 'mean'),
    mean_correctness=('correctness', 'mean'),
).sort_values('n_bonus', ascending=False).round(3)

print("\n── Bonus Claims by Approach ──")
print(bonus_by_approach.to_string())

# Contribution of bonus to total score
print("\n── Score Impact of Including Bonus Claims ──")
for approach in APPROACH_ORDER:
    for gps in [True, False]:
        if approach == 'Google Cloud Vision' and gps:
            continue
        mask = (df_entities['approach_name'] == approach) & (df_entities['use_gps'] == gps)
        all_scores = df_entities.loc[mask, 'entity_score']
        no_bonus = df_entities.loc[mask & (~df_entities['bonus']), 'entity_score']
        
        if len(all_scores) > 0 and len(no_bonus) > 0:
            diff = all_scores.mean() - no_bonus.mean()
            if abs(diff) > 0.001:
                label = f"{APPROACH_SHORT[approach]} {'GPS' if gps else 'No GPS'}" if approach != 'Google Cloud Vision' else 'GCV'
                print(f"  {label}: Δ = {diff:+.4f}")


---
## Section 18: Evaluator Notes Deeper Analysis

In [ ]:
# ── Images with evaluator notes: performance patterns ──
noted_imgs = df_images[df_images['evaluator_notes'].str.len() > 5]
print(f"Images with substantive notes: {len(noted_imgs)}")

# Difficulty distribution of noted images
if len(noted_imgs) > 0:
    print("\nDifficulty distribution of noted images:")
    print(noted_imgs['difficulty_without_gps'].value_counts().to_string())
    
    # Are noted images harder?
    noted_ids = set(noted_imgs['image_id'])
    noted_scores = df_evals[df_evals['image_id'].isin(noted_ids)]['overall_score']
    other_scores = df_evals[~df_evals['image_id'].isin(noted_ids)]['overall_score']
    
    print(f"\nMean score (noted): {noted_scores.mean():.4f}")
    print(f"Mean score (other): {other_scores.mean():.4f}")
    
    # Sample notes
    print("\n── Sample Evaluator Notes ──")
    for _, row in noted_imgs.head(5).iterrows():
        note = row['evaluator_notes'][:200]
        print(f"  Image {row['image_id']}: {note}...")


---
## Section 19: Summary and Recommendations

In [ ]:
# ── Final ranking ──
print("="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

# Overall ranking
ranking = df_evals.groupby(['approach_name', 'use_gps']).agg(
    overall=('overall_score', 'mean'),
    entity=('entity_score_mean', 'mean'),
    location=('location_score_mean', 'mean'),
    desc=('desc_score', 'mean'),
    precision=('entity_precision', 'mean'),
    recall=('entity_recall', 'mean'),
    f1=('entity_f1', 'mean'),
).round(4)

ranking = ranking.sort_values('overall', ascending=False)

# Add labels
labels = []
for (a, g) in ranking.index:
    s = APPROACH_SHORT[a]
    if a == 'Google Cloud Vision':
        labels.append('GCV')
    else:
        labels.append(f"{s} {'GPS' if g else 'No GPS'}")
ranking.index = labels

print("\n── Overall Ranking ──")
print(ranking.to_string())


In [ ]:
# ── Statistical significance summary ──
print("\n── Statistical Significance Summary ──")
print(f"\n1. FRIEDMAN TEST (overall differences among approaches):")
if len(pivot) > 0:
    stat, p = friedmanchisquare(*[pivot[col].values for col in pivot.columns])
    print(f"   χ² = {stat:.2f}, p = {p:.2e} {'*** SIGNIFICANT' if p < ALPHA else 'not significant'}")

print(f"\n2. GPS EFFECT (Wilcoxon signed-rank, overall):")
if len(common) > 0:
    stat, p = wilcoxon(gps_on.loc[common], gps_off.loc[common])
    print(f"   W = {stat:.0f}, p = {p:.2e} {'*** SIGNIFICANT' if p < ALPHA else 'not significant'}")

print(f"\n3. STRONG vs FAST (per model family):")
for _, row in df_sf.iterrows():
    sig = '***' if row['significant'] else 'n.s.'
    print(f"   {row['model_family']} ({row['gps']}): Δ = {row['delta_mean']:+.4f}, p = {row['wilcoxon_p']:.4f} {sig}")

print(f"\n4. ENTITY TYPE VARIATION (Kruskal-Wallis):")
if len(type_groups) >= 3:
    stat, p = kruskal(*type_groups)
    print(f"   H = {stat:.2f}, p = {p:.2e} {'*** SIGNIFICANT' if p < ALPHA else 'not significant'}")


In [ ]:
# ── Recommendation matrix ──
print("\n" + "="*70)
print("RECOMMENDATION MATRIX")
print("="*70)

# Best for each metric
metrics = {
    'Overall Score': 'overall',
    'Entity Detection': 'entity',
    'Location Accuracy': 'location',
    'Precision (fewer false positives)': 'precision',
    'Recall (fewer misses)': 'recall',
    'F1 Score': 'f1',
}

for metric_name, col in metrics.items():
    if col in ranking.columns:
        best = ranking[col].idxmax()
        best_val = ranking.loc[best, col]
        print(f"  Best {metric_name:40s}: {best:25s} ({best_val:.4f})")

print("\n── Key Findings ──")
print("1. GPS metadata effect on overall performance")
print("2. Strong vs fast model trade-off characteristics")  
print("3. Entity type-specific strengths per approach")
print("4. Confidence calibration varies significantly across approaches")
print("5. GCV provides complementary entity detection without description capability")

print("\n✓ Analysis complete.")
